# Fase 5 — Interpretação e Experimentos

## 🎯 Objetivo
Desenvolver intuição sobre o comportamento de redes neurais através de experimentos práticos: visualizar o que a rede aprendeu, comparar escolhas de design e entender overfitting.

Ao final deste notebook você será capaz de:
- **Plotar os hiperplanos** aprendidos pela camada oculta e comparar com a inicialização manual
- **Comparar ReLU vs Sigmoid vs LeakyReLU** em termos de convergência e acurácia
- **Variar hiperparâmetros** (learning rate, epochs, nº de neurônios) e observar o impacto
- **Identificar overfitting** e entender a diferença entre acurácia de treino e de teste

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

np.random.seed(42)
torch.manual_seed(42)

# === Dados (mesmos do notebook original) ===
N = 200
c1 = (0.90 + 0.25 * np.random.rand(N), 0.90 + 0.25 * np.random.rand(N))
c2 = (0.60 + 0.25 * np.random.rand(N), 0.60 + 0.25 * np.random.rand(N))
c3 = (0.30 + 0.25 * np.random.rand(N), 0.30 + 0.25 * np.random.rand(N))

clusters = (c1, c2, c3)
colors   = ("red", "green", "blue")
classes  = ("class1", "class2", "class3")

X = np.column_stack([
    np.concatenate([c[0] for c in clusters]),
    np.concatenate([c[1] for c in clusters])
]).astype(np.float32)
Y = np.array([0]*N + [1]*N + [2]*N)

# Divisão treino/teste
from sklearn.model_selection import train_test_split
X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.5, random_state=42, stratify=Y)

X_train = torch.FloatTensor(X_tr); Y_train = torch.LongTensor(Y_tr)
X_test  = torch.FloatTensor(X_te); Y_test  = torch.LongTensor(Y_te)

print(f"Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras")

## Experimento 1: Visualizando os Hiperplanos Aprendidos

Após o treino, podemos extrair os pesos da camada oculta e plotar as retas que cada neurônio aprendeu. Isso nos permite ver **onde a rede colocou suas fronteiras** e comparar com a inicialização geométrica ideal.

In [ ]:
# === Função auxiliar: treinar e retornar modelo ===
def treinar(ativacao, lr=0.1, epochs=150, n_hid=4, seed=42):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(2, n_hid),
        ativacao,
        nn.Linear(n_hid, 3)
    )
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight); m.bias.data.zero_()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    losses, accs = [], []

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss   = criterion(y_pred, Y_train)
        loss.backward(); optimizer.step()
        acc = (y_pred.argmax(1) == Y_train).float().mean().item()
        losses.append(loss.item()); accs.append(acc)

    model.eval()
    with torch.no_grad():
        y_te  = model(X_test)
        loss_te = criterion(y_te, Y_test).item()
        acc_te  = (y_te.argmax(1) == Y_test).float().mean().item()

    return model, losses, accs, loss_te, acc_te

# Treinar com ReLU
model_relu, losses_relu, accs_relu, loss_te_relu, acc_te_relu = treinar(nn.ReLU())

# Extrair pesos da camada oculta
pesos = model_relu[0].weight.data.numpy()  # shape (4, 2)
biases = model_relu[0].bias.data.numpy()   # shape (4,)

# Pesos da inicialização geométrica ideal
v  = (2.0**0.5) / 2
b1 = ((0.85)**2 + (0.85)**2)**0.5
b2 = ((0.55)**2 + (0.55)**2)**0.5
pesos_ideal  = np.array([[v,v], [-v,-v], [v,v], [-v,-v]])
biases_ideal = np.array([-b1, b1, -b2, b2])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

def plot_hiperplanos(ax, pesos, biases, titulo, neuron_colors=['purple','orange','brown','pink']):
    for cls, cor in zip(range(3), ['red','green','blue']):
        mask = Y_tr == cls
        ax.scatter(X_tr[mask,0], X_tr[mask,1], c=cor, alpha=0.4, s=20)

    x1r = np.linspace(0.1, 1.3, 300)
    for i, (w, b, cor) in enumerate(zip(pesos, biases, neuron_colors)):
        w1, w2 = w
        if abs(w2) > 1e-6:
            x2l = -(w1 * x1r + b) / w2
            mask_vis = (x2l > 0.05) & (x2l < 1.3)
            ax.plot(x1r[mask_vis], x2l[mask_vis], color=cor, lw=2.5,
                    label=f'N{i+1}: [{w1:.2f},{w2:.2f}] b={b:.2f}')
    ax.set_xlim(0.1, 1.3); ax.set_ylim(0.1, 1.3)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_title(titulo, fontsize=11)

plot_hiperplanos(ax1, pesos,        biases,        'Hiperplanos APRENDIDOS (ReLU)')
plot_hiperplanos(ax2, pesos_ideal,  biases_ideal,  'Hiperplanos IDEAIS (geométricos)')

plt.suptitle('Comparando: o que a rede aprendeu vs o que é ótimo', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Teste com ReLU: Loss={loss_te_relu:.4f}  Acurácia={acc_te_relu:.4f}")
print()
print("Pesos aprendidos:")
for i, (w, b) in enumerate(zip(pesos, biases)):
    print(f"  Neurônio {i+1}: w=[{w[0]:.3f},{w[1]:.3f}]  b={b:.3f}")
print("\nPesos ideais:")
for i, (w, b) in enumerate(zip(pesos_ideal, biases_ideal)):
    print(f"  Neurônio {i+1}: w=[{w[0]:.3f},{w[1]:.3f}]  b={b:.3f}")

## Experimento 2: Comparando Funções de Ativação

Aqui testamos ReLU, Sigmoid e LeakyReLU no mesmo problema e comparamos:
- **Velocidade de convergência** (quantas épocas para atingir boa acurácia)
- **Acurácia final** no treino e no teste
- **Onde as fronteiras ficaram**

In [ ]:
# === Comparação: ReLU vs Sigmoid vs LeakyReLU ===
configuracoes = [
    ('ReLU',        nn.ReLU(),                    'red'   ),
    ('Sigmoid',     nn.Sigmoid(),                 'green' ),
    ('LeakyReLU',   nn.LeakyReLU(0.01),           'blue'  ),
]

resultados = {}
for nome, ativ, cor in configuracoes:
    model, losses, accs, loss_te, acc_te = treinar(ativ, lr=0.1, epochs=200)
    resultados[nome] = {
        'model': model, 'losses': losses, 'accs': accs,
        'loss_te': loss_te, 'acc_te': acc_te, 'cor': cor
    }
    print(f"{nome:12s}: Treino acc={accs[-1]:.4f}  Teste acc={acc_te:.4f}  Teste loss={loss_te:.4f}")

# Plot comparativo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Curvas de loss
for nome, res in resultados.items():
    axes[0].plot(res['losses'], color=res['cor'], lw=2, label=nome)
axes[0].set_title('Curva de Loss'); axes[0].set_xlabel('Época')
axes[0].set_ylabel('CrossEntropy Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Curvas de acurácia
for nome, res in resultados.items():
    axes[1].plot(res['accs'], color=res['cor'], lw=2, label=nome)
axes[1].set_title('Acurácia no Treino'); axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Acurácia final treino vs teste
nomes_list = list(resultados.keys())
acc_tr_list = [resultados[n]['accs'][-1] for n in nomes_list]
acc_te_list = [resultados[n]['acc_te']    for n in nomes_list]
x_pos = np.arange(len(nomes_list))
w = 0.35
axes[2].bar(x_pos - w/2, acc_tr_list, w, label='Treino', alpha=0.8)
axes[2].bar(x_pos + w/2, acc_te_list, w, label='Teste',  alpha=0.8)
axes[2].set_xticks(x_pos); axes[2].set_xticklabels(nomes_list)
axes[2].set_ylabel('Acurácia'); axes[2].set_title('Treino vs Teste por Ativação')
axes[2].legend(); axes[2].grid(True, alpha=0.3); axes[2].set_ylim(0.7, 1.05)
for i, (tr, te) in enumerate(zip(acc_tr_list, acc_te_list)):
    axes[2].text(i-w/2, tr+0.005, f'{tr:.3f}', ha='center', fontsize=9)
    axes[2].text(i+w/2, te+0.005, f'{te:.3f}', ha='center', fontsize=9)

plt.suptitle('Comparação de Funções de Ativação', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Experimento 3: Variando Hiperparâmetros

### Hiperparâmetros mais importantes:

| Hiperparâmetro | Efeito se muito baixo | Efeito se muito alto |
|---|---|---|
| **Learning rate** | Converge devagar | Oscila, pode divergir |
| **Epochs** | Underfitting | Overfitting |
| **Nº de neurônios** | Underfitting (pouca capacidade) | Overfitting + lento |

In [ ]:
# === Variando o Learning Rate ===
learning_rates = [0.001, 0.01, 0.1, 0.5, 2.0]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for lr in learning_rates:
    try:
        _, losses, accs, loss_te, acc_te = treinar(nn.ReLU(), lr=lr, epochs=200)
        losses_clean = [min(l, 5.0) for l in losses]  # limitar para visualização
        axes[0].plot(losses_clean, lw=2, label=f'lr={lr}')
        axes[1].plot(accs, lw=2, label=f'lr={lr} (teste={acc_te:.2f})')
    except:
        print(f"lr={lr}: divergiu")

axes[0].set_title('Loss por Learning Rate'); axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss'); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Acurácia por Learning Rate'); axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia'); axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
plt.suptitle('Impacto do Learning Rate', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Experimento 4: Overfitting — Treino vs Teste

**Overfitting** ocorre quando a rede "memoriza" os dados de treino em vez de aprender padrões generalizáveis. Isso se manifesta como:
- Acurácia de treino → 100%
- Acurácia de teste → bem menor

**Como detectar:** a curva de loss no teste começa a subir enquanto no treino continua caindo.

**Causas comuns:**
- Rede muito grande para poucos dados
- Treino por muitas épocas
- Learning rate muito baixo (rede se especializa demais nos exemplos)

In [ ]:
# === Demonstração de Overfitting com rede muito grande ===
def treinar_monitorado(n_hid, lr=0.05, epochs=500, seed=42):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(2, n_hid), nn.ReLU(),
        nn.Linear(n_hid, n_hid), nn.ReLU(),   # segunda camada oculta
        nn.Linear(n_hid, 3)
    )
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight); m.bias.data.zero_()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses_tr, accs_tr, losses_te, accs_te = [], [], [], []

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss   = criterion(y_pred, Y_train)
        loss.backward(); optimizer.step()
        losses_tr.append(loss.item())
        accs_tr.append((y_pred.argmax(1) == Y_train).float().mean().item())

        model.eval()
        with torch.no_grad():
            y_te   = model(X_test)
            loss_t = criterion(y_te, Y_test).item()
            acc_te = (y_te.argmax(1) == Y_test).float().mean().item()
        losses_te.append(loss_t); accs_te.append(acc_te)

    return losses_tr, accs_tr, losses_te, accs_te

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for col, (n_hid, titulo) in enumerate([(4, 'Rede pequena (4 neurônios)
→ Sem overfitting'),
                                        (64, 'Rede grande (64 neurônios)
→ Overfitting!')]):
    lt, at, lte, ate = treinar_monitorado(n_hid)

    axes[0, col].plot(lt,  'r-', lw=2, label='Treino')
    axes[0, col].plot(lte, 'b--', lw=2, label='Teste')
    axes[0, col].set_title(f'Loss — {titulo}', fontsize=10)
    axes[0, col].set_xlabel('Época'); axes[0, col].set_ylabel('Loss')
    axes[0, col].legend(); axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(at,  'r-', lw=2, label='Treino')
    axes[1, col].plot(ate, 'b--', lw=2, label='Teste')
    axes[1, col].set_title(f'Acurácia — {titulo}', fontsize=10)
    axes[1, col].set_xlabel('Época'); axes[1, col].set_ylabel('Acurácia')
    axes[1, col].legend(); axes[1, col].grid(True, alpha=0.3)

    print(f"n_hid={n_hid:2d}: Treino={at[-1]:.4f}  Teste={ate[-1]:.4f}  "
          f"Gap={at[-1]-ate[-1]:.4f}  {'⚠ OVERFITTING' if at[-1]-ate[-1] > 0.05 else '✓ OK'}")

plt.suptitle('Overfitting: Rede pequena vs. Rede grande', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print()
print("Sinais de overfitting:")
print("  - Loss de TREINO cai, mas loss de TESTE sobe ou estagna")
print("  - Grande diferença entre acurácia de treino e de teste")